# Camada Silver
Nessa etapa trataremos as tabelas da etapa bronze, definindo tipos, lidando com nulos, eliminando dados invalidos e demais transformações necessárias

## VRA
Vamos inspecionar a tabela `mvp_engenharia_dados.bronze.bronze_vra` para determinar quais tratamentos serão necessàrios

In [0]:
%sql
DESCRIBE mvp_engenharia_dados.bronze.bronze_vra

### Duplicidade de dados

In [0]:
from pyspark.sql.functions import col

vra = spark.table("mvp_engenharia_dados.bronze.bronze_vra")

total_rows = vra.count()
distinct_rows = vra.distinct().count()
duplicates = total_rows - distinct_rows

print(f"Total de linhas: {total_rows:,d}")
print(f"Linhas distintas: {distinct_rows:,d}")
print(f"Duplicatas: {duplicates:,d} ({duplicates / total_rows * 100:.2f}%)")

if duplicates > 0:
    print("\nTop 10 linhas duplicadas:")
    display(
        vra.groupBy(vra.columns)
          .count()
          .filter(col("count") > 1)
          .orderBy(col("count").desc())
          .limit(10)
    )

In [0]:
vra = vra.drop_duplicates()

### Valores Nulos ou em Branco

In [0]:
from pyspark.sql.functions import col, when, count, trim

total = vra.count()

null_row = vra.select([count(when(col(c).isNull(), c)).alias(c) for c in vra.columns]).collect()[0]
blank_row = vra.select([count(when((col(c).isNotNull()) & (trim(col(c)) == ""), c)).alias(c) for c in vra.columns]).collect()[0]

results = [(c, int(null_row[c]), int(blank_row[c])) for c in vra.columns]
display(spark.createDataFrame(results, ["column", "null_count", "blank_count"]))

> Nao ha valores em branco (''), somente nulos, o que simplifica o tratamento.

#### Tipo Linha nulos

In [0]:
%sql
select * from mvp_engenharia_dados.bronze.bronze_vra where `Código Tipo Linha` is null

> Voos com Código Tipo Linha em branco estao todos **CANCELADOS**.  
> Como de qualquer forma esses voos nao serao considerados, podemos ignora-los 

In [0]:
vra = vra.dropna(subset="Código Tipo Linha")

#### Partida Prevista nulos

In [0]:
display(vra.filter(col("Partida Prevista").isNull()).limit(25))

> Os voos sem Partida Prevista (Ou chegada Prevista), não devem ser contabilizados para o fim de pontualidade.  
> Eles distorceriam o índice, melhorando o resultado artificialmente

In [0]:
vra = vra.dropna(subset=["Partida Prevista", "Chegada Prevista"], how='any')

#### Partida Real e Chegada Real
Vamos verificar se as partidas realizadas obrigatoriamente tem chegadas realizadas (e vice versa)

In [0]:
%sql
SELECT
	*

FROM
	mvp_engenharia_dados.bronze.bronze_vra

WHERE
	(
		`Partida Real` IS NULL
		AND `Chegada Real` IS NOT NULL
	)
	OR (
		`Partida Real` IS NOT NULL
		AND `Chegada Real` IS NULL
	)

> As partidas e chegadas realizadas sempre possuem par, ou seja, nunca temos somente a partida ou a chegada registrada somente, sempre temos ambas.
> Podemos filtrar tranquilamente as viagens sem partidas real registrada

In [0]:
vra = vra.dropna(subset=["Partida Real", "Chegada Real"], how='any')

### Justificativa
O campo Justificativa deixou de ser obrigatorio a partir de 2020, por isso está totalmente nulo na base.  
Vamos eliminar essa coluna

In [0]:
vra = vra.drop("Justificativa")

### Codeshare
Não é um campo relevante para nossa analise, então tambem pode ser eliminado

In [0]:
vra = vra.drop("Codeshare")

### Nulos pos tratamento

In [0]:
total_anterior = spark.table("mvp_engenharia_dados.bronze.bronze_vra").count()
total = vra.count()

null_row = vra.select([count(when(col(c).isNull(), c)).alias(c) for c in vra.columns]).collect()[0]
blank_row = vra.select([count(when((col(c).isNotNull()) & (trim(col(c)) == ""), c)).alias(c) for c in vra.columns]).collect()[0]

results = [(c, int(null_row[c]), int(blank_row[c])) for c in vra.columns]
display(spark.createDataFrame(results, ["column", "null_count", "blank_count"]))

print(f"Total de linhas Antes: {total_anterior}")
print(f"Total de linhas Atual: {total}")
print(f"Percentual Perdido: {(total_anterior - total) / total_anterior * 100:.2f}%")


> Sem mais nenhum valor nulo nas colunas.
> Foram eliminados um total de 7% de linhas

## Filtros de regras

Como só vamos considerar voos nacionais e não cancelados nas nossas métricas, podemos eliminar os demais voos da tabela.

### Voos Cancelados

In [0]:
vra.filter(col("Situação Voo") == "CANCELADO").count()

> Os filtros de nulo anteriores acabaram já eliminando os Cancelados

In [0]:
display(vra.groupBy("Situação Voo").count())

> Ao meu ver não faz sentido manter essa coluna se a tabela só irá possuir voos realizados

In [0]:
vra = vra.drop("Situação Voo")

### Tipos de Linhas
Como só queremos as linhas domésticas de passageiros, usaremos o tipo da linha para determinar.

| Código | Descrição |
| --- | --- |
| **N** | **Doméstica Mista: transporte de passageiros ou mistos, com todos os aeródromos em território brasileiro** |
| C | Doméstica Cargueira: transporte exclusivo de carga e/ou mala postal, com todos os aeródromos em território brasileiro |
| I | Internacional Mista: transporte de passageiros ou mistos, com ao menos um aeródromo em território estrangeiro |
| G | Internacional Cargueira: transporte exclusivo de carga e/ou rede postal, com ao menos um aeródromo em território estrangeiro |
| X | Não informado |

O que interessa para a nossa análise são as linhas do tipo **N**

In [0]:
display(vra.groupBy(col("Código Tipo Linha")).count())

In [0]:
vra =  vra.filter(col("Código Tipo Linha") == 'N')


E agora o campo se torna obsoleto

In [0]:
vra = vra.drop("Código Tipo Linha")

In [0]:
display(vra.limit(10))

## Código DI
O Código DI (Dígito Identificador) informa o tipo de etapa de voo.

| Código DI | Descrição |
| --- | --- |
| 0 | Etapa Regular |
| 2 | Etapa Extra |
| 3 | Etapa de Retorno |
| 4 | Inclusão de Etapa |
| 6 | Etapa Não Remunerada Sem Transporte de Objetos |
| 7 | Etapa de Voo de Fretamento |
| 9 | Etapa de Voo Charter |
| D | Etapa de Voo Duplicada |
| E | Etapa Não Remunerada Com Transporte de Objetos |

In [0]:
display(vra.groupBy("Código DI").count().sort(col("count").desc()))

In [0]:
from pyspark.sql.functions import when, lit

vra = vra.withColumn(
    "descricao_etapa",
    when(col("Código DI") == "0", lit("Etapa Regular"))
    .when(col("Código DI") == "2", lit("Etapa Extra"))
    .when(col("Código DI") == "3", lit("Etapa de Retorno"))
    .when(col("Código DI") == "4", lit("Inclusão de Etapa"))
    .when(col("Código DI") == "6", lit("Etapa Não Remunerada Sem Transporte de Objetos"))
    .when(col("Código DI") == "7", lit("Etapa de Voo de Fretamento"))
    .when(col("Código DI") == "9", lit("Etapa de Voo Charter"))
    .when(col("Código DI") == "D", lit("Etapa de Voo Duplicada"))
    .when(col("Código DI") == "E", lit("Etapa Não Remunerada Com Transporte de Objetos"))
    .otherwise(lit("Não identificado"))
)

display(vra.groupBy("Código DI", "descricao_etapa").count().orderBy(col("count").desc()))

Vamos manter apenas a descricao_etapa, e não mais o código

In [0]:
vra = vra.drop("Código DI")

## Campos que virão de dimensões
Vamos eliminar da tabela vra os campos que serão provenientes de tabelas de dimensão, como informaçao de empresa e aeroporto

In [0]:
vra = vra.drop("Empresa Aérea", "Descrição Aeroporto Origem", "Descrição Aeroporto Destino")

In [0]:
display(vra.limit(10))

## Normalizar os nomes das colunas da tabela
Utilizaremos a convencao snake_case

In [0]:
import re
import unicodedata

def to_snake_case(name):
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if not unicodedata.combining(c))
    name = re.sub(r'[^a-zA-Z0-9]', '_', name).lower()
    name = re.sub(r'_+', '_', name).strip('_')
    return name

In [0]:
for old_name in vra.columns:
    new_name = to_snake_case(old_name)
    if new_name != old_name:
        vra = vra.withColumnRenamed(old_name, new_name)

display(vra.limit(10))

## Tipos de dados
Vamos transformar para os tipos corretos. (datas, numeros...etc)

### Tipos numéricos
Só sobrou o numero_de_asentos como numérico, numero_voo por exemplo contem caracteres alfabéticos

In [0]:
vra = vra.withColumn("numero_de_assentos", col("numero_de_assentos").cast("int"))
display(vra.select("numero_de_assentos").describe())

> vale investigar esses voos com 0 assentos

In [0]:
display(vra.filter(col("numero_de_assentos") == 0).groupBy("descricao_etapa").count())

> Um misto de tipos de voos.
> Irei manter com a seguinte premissa: Esses voos embora não estejam com passageiros, podem afetar etapas subsequentes.

### Tipos de data hora e data

In [0]:
from pyspark.sql.functions import to_timestamp, to_date, try_to_date, coalesce, col

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dtypes = dict(vra.dtypes)
if dtypes.get("partida_prevista") == "string":
    vra = vra.withColumn("partida_prevista", to_timestamp("partida_prevista", "dd/MM/yyyy HH:mm"))
if dtypes.get("partida_real") == "string":
    vra = vra.withColumn("partida_real", to_timestamp("partida_real", "dd/MM/yyyy HH:mm"))
if dtypes.get("chegada_prevista") == "string":
    vra = vra.withColumn("chegada_prevista", to_timestamp("chegada_prevista", "dd/MM/yyyy HH:mm"))
if dtypes.get("chegada_real") == "string":
    vra = vra.withColumn("chegada_real", to_timestamp("chegada_real", "dd/MM/yyyy HH:mm"))
if dtypes.get("referencia") == "string":
    vra = vra.withColumn("referencia", coalesce(try_to_date(col("referencia"), "dd/MM/yyyy HH:mm:ss"), try_to_date(col("referencia"), "yyyy-MM-dd")))

display(vra.limit(10))

## Novas colunas
Vamos adicionar novas colunas que podem ser uteis a analise, como data de partida/chegada e tempo de atraso

### Datas de partida e chegada

In [0]:
from pyspark.sql.functions import to_date

vra = (vra
    .withColumn("data_partida", to_date("partida_prevista"))
    .withColumn("data_chegada", to_date("chegada_prevista"))
)

display(vra.limit(10))

### Tempo de atraso (partida e chegada)

In [0]:
from pyspark.sql.functions import unix_timestamp, round

vra = (vra
    .withColumn("atraso_partida_min", round((unix_timestamp("partida_real") - unix_timestamp("partida_prevista")) / 60))
    .withColumn("atraso_chegada_min", round((unix_timestamp("chegada_real") - unix_timestamp("chegada_prevista")) / 60))
)

display(vra.limit(10))

## Categorias
Uma verificacao na situacao_partida e situacao_chegada, para confirmar que estao padronizadas

In [0]:
display(vra.groupBy("situacao_partida").count())
display(vra.groupBy("situacao_chegada").count())


> As categorias já seguem uma nomenclatura padronizada

## Persistir os dados da vra no schema silver

In [0]:
vra.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("mvp_engenharia_dados.silver.silver_vra")

print(f"Tabela silver_vra criada com {vra.count():,d} linhas e {len(vra.columns)} colunas")

In [0]:
%sql
select * from mvp_engenharia_dados.silver.silver_vra limit 100

## Aeroportos

In [0]:
aeroportos = spark.table("mvp_engenharia_dados.bronze.bronze_aerodromos")

display(aeroportos.limit(10))

### Verificacao de duplicidade

In [0]:
from pyspark.sql.functions import count as count_col, col

total_linhas = aeroportos.count()

duplicatas_completas = aeroportos.groupBy(aeroportos.columns).agg(count_col("*").alias("qtd")).filter(col("qtd") > 1)
qtd_duplicatas_completas = duplicatas_completas.count()
linhas_duplicadas_completas = duplicatas_completas.agg({"qtd": "sum"}).collect()[0][0] or 0

# OACI é o identificador internacional; CIAD é o identificador civil brasileiro — ambos devem ser únicos
duplicatas_oaci = aeroportos.groupBy("Código OACI").agg(count_col("*").alias("qtd")).filter(col("qtd") > 1)
qtd_duplicatas_oaci = duplicatas_oaci.count()
linhas_duplicadas_oaci = duplicatas_oaci.agg({"qtd": "sum"}).collect()[0][0] or 0

duplicatas_ciad = aeroportos.groupBy("CIAD").agg(count_col("*").alias("qtd")).filter(col("qtd") > 1)
qtd_duplicatas_ciad = duplicatas_ciad.count()
linhas_duplicadas_ciad = duplicatas_ciad.agg({"qtd": "sum"}).collect()[0][0] or 0

print(f"Total de linhas: {total_linhas:,d}")
print(f"\n--- Duplicatas (todas as colunas idênticas) ---")
print(f"Grupos duplicados: {qtd_duplicatas_completas}")
print(f"Linhas envolvidas: {linhas_duplicadas_completas} ({linhas_duplicadas_completas/total_linhas*100:.2f}% do total)")
print(f"\n--- Duplicatas por Código OACI ---")
print(f"Códigos OACI duplicados: {qtd_duplicatas_oaci}")
print(f"Linhas envolvidas: {linhas_duplicadas_oaci} ({linhas_duplicadas_oaci/total_linhas*100:.2f}% do total)")
print(f"\n--- Duplicatas por CIAD ---")
print(f"CIADs duplicados: {qtd_duplicatas_ciad}")
print(f"Linhas envolvidas: {linhas_duplicadas_ciad} ({linhas_duplicadas_ciad/total_linhas*100:.2f}% do total)")

if qtd_duplicatas_oaci > 0:
    print("\n--- Exemplos de duplicatas por Código OACI ---")
    display(duplicatas_oaci.orderBy(col("qtd").desc()).limit(20))

> Não há duplicidade nessa tabela

In [0]:
from pyspark.sql.functions import col, count as count_col

import builtins
total_linhas = aeroportos.count()

# Vazios ('') são contados separadamente porque o CSV da ANAC pode usar string vazia em vez de NULL
resultado = []
for c in aeroportos.columns:
    nulls = aeroportos.filter(col(c).isNull()).count()
    vazios = aeroportos.filter((col(c) == "") | (col(c) == " ")).count()
    total_nulos = nulls + vazios
    pct = builtins.round(total_nulos / total_linhas * 100, 2)
    resultado.append((c, nulls, vazios, total_nulos, float(pct)))

schema = "coluna string, nulos_null int, nulos_vazio_str int, total_nulos int, pct_nulos double"
df_nulos = spark.createDataFrame(resultado, schema=schema)
display(df_nulos.orderBy(col("total_nulos").desc()))

> Temos nulos mas não temos nulos de string vazia.  
> Muitas das colunas de nulos não são relevantes a nossa analise final.

#### Nulos em Municipio

In [0]:
display(
    aeroportos.filter(col("Município").isNull()).select(
        "Código OACI",
        "CIAD",
        "Nome",
        "Município",
        "UF",
        "Município Servido",
        "UF Servido",
    )
)

> Nesses casos os nulos possuem o campo de Municipio Servido preenchidos. Acho razoavel utilizar esse campo como alternativa 

In [0]:
from pyspark.sql.functions import coalesce, col

aeroportos = (aeroportos
    .withColumn("Município", coalesce(col("Município"), col("Município Servido")))
    .withColumn("UF", coalesce(col("UF"), col("UF Servido")))
)

display(aeroportos.filter(col("Código OACI").isin("SBIZ", "SDJV", "SBME", "SBCR", "SDRS")).select("Código OACI", "Nome", "Município", "UF", "Município Servido", "UF Servido"))

#### Nulos em Municipio Servido e UF Servido

In [0]:
display(
    aeroportos.filter(col("Município Servido").isNull() | col("UF Servido").isNull()).select(
        "Código OACI", "CIAD", "Nome", "Município", "UF", "Município Servido", "UF Servido"
    )
)

> Mesma logica do tratamento anterior, só que no caminho inverso

In [0]:
aeroportos = (aeroportos
    .withColumn("Município Servido", coalesce(col("Município Servido"), col("Município")))
    .withColumn("UF Servido", coalesce(col("UF Servido"), col("UF")))
)

display(aeroportos.filter(col("Código OACI").isin("SBPW", "SSBG")).select("Código OACI", "Nome", "Município", "UF", "Município Servido", "UF Servido"))

In [0]:
import builtins
total_linhas = aeroportos.count()

resultado = []
for c in aeroportos.columns:
    nulls = aeroportos.filter(col(c).isNull()).count()
    vazios = aeroportos.filter((col(c) == "") | (col(c) == " ")).count()
    total_nulos = nulls + vazios
    pct = builtins.round(total_nulos / total_linhas * 100, 2)
    resultado.append((c, nulls, vazios, total_nulos, float(pct)))

schema = "coluna string, nulos_null int, nulos_vazio_str int, total_nulos int, pct_nulos double"
df_nulos = spark.createDataFrame(resultado, schema=schema)
display(df_nulos.orderBy(col("total_nulos").desc()))


> Como dito anteriormente, as colunas restantes com nulo não sao relevantes para a nossa analise e serão eliminadas

In [0]:
colunas_remover = ["Operação Noturna", "Operação Diurna", "Validade do Registro", "Portaria de Registro", "Link Portaria"]
aeroportos = aeroportos.drop(*colunas_remover)

print(f"Colunas removidas: {colunas_remover}")
print(f"Colunas restantes: {len(aeroportos.columns)}")
display(aeroportos.limit(5))

### Padronizacao de colunas

Vamos padronizar as escritas dos campos descritivos como Nome, Municipio etc...

In [0]:
from pyspark.sql.functions import upper

colunas_texto = ["Nome", "Município", "UF", "Município Servido", "UF Servido", "Situação"]
for c in colunas_texto:
    aeroportos = aeroportos.withColumn(c, upper(col(c)))

display(aeroportos.limit(10))

### Transformacao de Tipos

In [0]:
from pyspark.sql.functions import regexp_replace, col

# Altitude usa vírgula como separador decimal (padrão brasileiro)
aeroportos = aeroportos.withColumns({
    "LATGEOPOINT": col("LATGEOPOINT").cast("double"),
    "LONGEOPOINT": col("LONGEOPOINT").cast("double"),
    "Altitude": regexp_replace(col("Altitude"), ",", ".").cast("double"),
})

aeroportos.printSchema()
display(aeroportos.limit(10))

### Padronizacao de nomes de colunas

In [0]:
for old_name in aeroportos.columns:
    new_name = to_snake_case(old_name)
    if new_name != old_name:
        aeroportos = aeroportos.withColumnRenamed(old_name, new_name)

aeroportos.printSchema()
display(aeroportos.limit(10))

### Verificacao de valores das colunas
Como codigo_oaci e ciad foram verificados anteriormente na etapa de duplidade, não é necessario verifica novamente

In [0]:
display(
    aeroportos.groupBy("nome")
        .count()
        .orderBy(col("count").desc())
)

In [0]:
display(
    aeroportos.filter(col("nome").isin("MANOEL RIBAS", "SANTOS DUMONT", "SANTA MARIA"))
        .orderBy("nome", "uf")
)

> Os nomes duplicados são casos legitimos de aeroportos diferentes, que apenas compartilham o mesmo nome.

In [0]:
display(
    aeroportos.groupBy("municipio")
        .count()
        .orderBy(col("count").desc())
)

> Nada de notavel

In [0]:
display(
    aeroportos.groupBy("uf")
        .count()
        .orderBy(col("count").desc())
)

> Valores seguem um padrao consistente de nomenclatura dos estados

In [0]:
display(
    aeroportos.groupBy("municipio_servido")
        .count()
        .orderBy(col("count").desc())
)

In [0]:
display(
    aeroportos.groupBy("uf_servido")
        .count()
        .orderBy(col("count").desc())
)

> Novamente sem inconsistencias

In [0]:
from pyspark.sql.functions import min as min_col, max as max_col

display(
    aeroportos.agg(
        min_col("latgeopoint").alias("lat_min"),
        max_col("latgeopoint").alias("lat_max"),
        min_col("longeopoint").alias("lon_min"),
        max_col("longeopoint").alias("lon_max"),
    )
)

> Sem valores fora do range de latitude e longitude esperados

In [0]:
from pyspark.sql.functions import mean as mean_col

display(
    aeroportos.agg(
        min_col("altitude").alias("alt_min"),
        max_col("altitude").alias("alt_max"),
        mean_col("altitude").alias("alt_media"),
    )
)

> Valores de altitude dentro do esperado

In [0]:
display(
    aeroportos.groupBy("situacao")
        .count()
        .orderBy(col("count").desc())
)

> Existem aeroportos com a situacao INTERDITADO, mas devemos manter esses casos pois estamos lidando com uma base historica.

In [0]:
(aeroportos.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_aerodromos"))

print(f"Tabela silver_aerodromos criada com {aeroportos.count():,d} linhas")
display(spark.table("mvp_engenharia_dados.silver.silver_aerodromos").limit(5))

### Nome de Aeroporto
Como vamos querer ver analises agrupadas por aeroportos, quero criar um nome que seja reconhecivel mas ainda unico.
Iremos fazer isso juntando o codigo_oaci (ICAO) com o nome.
Exemplo: `PARINTINS (SWPI)`

In [0]:
from pyspark.sql.functions import concat, col, lit

aeroportos = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")

aeroportos = aeroportos.withColumn(
    "aerodromo",
    concat(col("nome"), lit(" ("), col("codigo_oaci"), lit(")"))
)

(aeroportos.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_aerodromos"))

print(f"Tabela silver_aerodromos atualizada com {aeroportos.count():,d} linhas")
display(spark.table("mvp_engenharia_dados.silver.silver_aerodromos").limit(10))

## Empresas

In [0]:
empresas = spark.table("mvp_engenharia_dados.bronze.bronze_empresas")

display(empresas.limit(10))

### Filtro de colunas nao utilizadas

Diferentemente de outros passos anteriores, vamos primeiro ja eliminar colunas que nao serao uteis a nossa analise, ou que não tenham um possivel uso futuro.

In [0]:
colunas_remover = ["CEP", "Data", "Email", "Endereco", "Instrumento", "Telefone", "Servico"]

empresas = empresas.drop(*colunas_remover)

print(f"Colunas restantes: {empresas.columns}")
display(empresas.limit(10))

### ICAO
ICAO será nossa chave para a tabela VRA, empresas sem esse codigo não são parte do nosso escopo

In [0]:
from pyspark.sql.functions import trim

empresas = empresas.filter(
    col("ICAO").isNotNull() & (trim(col("ICAO")) != "")
)

print(f"Empresas com ICAO válido: {empresas.count():,d}")
display(empresas.limit(10))

> Isso eliminou boa parte das empresas da nossa base, o que ja era esperado. Afinal temos poucas empresas atuando em voos apenas domesticos.

### Verificacao de Nulos

In [0]:
from pyspark.sql.functions import col, count as count_col

import builtins
total_linhas = empresas.count()

# Vazios ('') são contados separadamente porque o CSV da ANAC pode usar string vazia em vez de NULL
resultado = []
for c in empresas.columns:
    nulls = empresas.filter(col(c).isNull()).count()
    vazios = empresas.filter((col(c) == "") | (col(c) == " ")).count()
    total_nulos = nulls + vazios
    pct = builtins.round(total_nulos / total_linhas * 100, 2)
    resultado.append((c, nulls, vazios, total_nulos, float(pct)))

schema = "coluna string, nulos_null int, nulos_vazio_str int, total_nulos int, pct_nulos double"
df_nulos = spark.createDataFrame(resultado, schema=schema)
display(df_nulos.orderBy(col("total_nulos").desc()))

> Com o ICAO ja filtrado, não temos nulos relevantes nessa tabela, apenas no campo `Estrangeira`, que é o código que a empresa utiliza como identificador internacional

In [0]:
from pyspark.sql.functions import when, trim, lit

empresas = empresas.withColumn(
    "Estrangeira",
    when((col("Estrangeira").isNull()) | (trim(col("Estrangeira")) == ""), lit("-"))
    .otherwise(col("Estrangeira"))
)

display(empresas.filter(col("Estrangeira") == "-").limit(10))

### Duplicidade
Vamos verificar se existe dados duplicados na tabela e de codigo ICAO

In [0]:
# Duplicidade total (todas as colunas iguais)
total_duplicadas = empresas.count() - empresas.distinct().count()

# Duplicidade por ICAO (chave de negócio)
dups_icao = (empresas
    .groupBy("ICAO")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc()))

print(f"Linhas duplicadas (todas colunas): {total_duplicadas}")
print(f"ICAO duplicados: {dups_icao.count()}")
display(dups_icao)

> Não há duplicidade

### Verificacao e padronizacao de valores

#### Coluna `Ativa`

In [0]:
display(
    empresas.groupBy("Ativa")
        .count()
        .orderBy(col("count").desc())
)

> Apenas um valor unico, nada a padronizar.

#### Vamos padronizar os campos textuais em UPPERCASE

In [0]:
from pyspark.sql.functions import upper, trim

for c in empresas.columns:
    if dict(empresas.dtypes)[c] == "string":
        empresas = empresas.withColumn(c, trim(upper(col(c))))

display(empresas.limit(10))

#### Nome empresa
Semelhante ao que fizemos com os aerportos vamos criar um campo de nome da empresa com o nome curto e o codigo ICAO


In [0]:
from pyspark.sql.functions import split, concat, lit

empresas = empresas.withColumn(
    "empresa",
    concat(split(col("Razao"), " ").getItem(0), lit(" ("), col("ICAO"), lit(")"))
)

display(empresas.select("Razao", "ICAO", "empresa"))

> Nesse resultado é possivel notar a empresa VOAR com ICAO *** o que nao é valido

In [0]:
empresas = empresas.filter(col("ICAO") != "***")

print(f"Empresas após remover ICAO='***': {empresas.count():,d}")
display(empresas.select("Razao", "ICAO", "empresa"))

### Padronizacao dos campos
Vamos padronizar as colunas como snake_case


In [0]:
for old_name in empresas.columns:
    new_name = to_snake_case(old_name)
    if new_name != old_name:
        empresas = empresas.withColumnRenamed(old_name, new_name)

display(empresas.limit(10))

### Persistir os dados no schema silver


In [0]:
(empresas.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.silver.silver_empresas"))

print(f"Tabela silver_empresas criada com {empresas.count():,d} linhas")
display(spark.table("mvp_engenharia_dados.silver.silver_empresas").limit(10))

## Resumo do Notebook

Este notebook transformou as tabelas da camada **bronze** em tabelas tratadas na camada **silver**, aplicando limpeza, padronizacao, filtros de qualidade e tipagem adequada. Foram processadas tres tabelas:

### 1. `silver_vra` (tabela fato - voos)
- **Origem:** `mvp_engenharia_dados.bronze.bronze_vra`
- **Resultado:** 2.736.859 linhas e 18 colunas
- **Tratamentos aplicados:**
  - Remocao de duplicatas
  - Eliminacao de voos cancelados e sem datas previstas/reais (perda de ~7%)
  - Filtro apenas para linhas domesticas de passageiros (`Codigo Tipo Linha` = 'N')
  - Remocao de colunas nao relevantes: `Justificativa`, `Codeshare`, `Situacao Voo`, `Codigo Tipo Linha`
  - Criacao de `descricao_etapa` a partir do mapeamento do `Codigo DI`
  - Remocao de campos descritivos que virao de tabelas dimensao (`Empresa Aerea`, `Descricao Aeroporto Origem/Destino`)
  - Conversao de tipos: `numero_de_assentos` para inteiro, datas para `timestamp`/`date` (timezone America/Sao_Paulo)
  - Criacao de colunas derivadas: `data_partida`, `data_chegada`, `atraso_partida_min`, `atraso_chegada_min`
  - Padronizacao snake_case em todos os nomes de coluna
  - Verificacao das categorias de `situacao_partida` e `situacao_chegada` (ja padronizadas)

### 2. `silver_aerodromos` (dimensao - aeroportos)
- **Origem:** `mvp_engenharia_dados.bronze.bronze_aerodromos`
- **Resultado:** 496 linhas e 14 colunas
- **Tratamentos aplicados:**
  - Verificacao de duplicidade (nenhuma encontrada por OACI, CIAD ou linha completa)
  - Tratamento de nulos: `Municipio`/`UF` preenchidos com `Municipio Servido`/`UF Servido` e vice-versa via `coalesce`
  - Remocao de colunas administrativas com nulos: `Operacao Noturna`, `Operacao Diurna`, `Validade do Registro`, `Portaria de Registro`, `Link Portaria`
  - Padronizacao UPPERCASE em campos textuais
  - Conversao de tipos: `latgeopoint`/`longeopoint` para double, `altitude` para double (correcao de virgula decimal)
  - Padronizacao snake_case nos nomes de coluna
  - Criacao da coluna `aerodromo` no formato `NOME (CODIGO_OACI)`, ex: `PARINTINS (SWPI)`

### 3. `silver_empresas` (dimensao - empresas aereas)
- **Origem:** `mvp_engenharia_dados.bronze.bronze_empresas`
- **Resultado:** 19 linhas e 7 colunas
- **Tratamentos aplicados:**
  - Remocao de colunas nao utilizadas: `CEP`, `Data`, `Email`, `Endereco`, `Instrumento`, `Telefone`, `Servico`
  - Filtro apenas para empresas com codigo `ICAO` valido (nao nulo e nao vazio)
  - Preenchimento do unico valor vazio em `Estrangeira` com `-`
  - Verificacao de duplicidade (nenhuma encontrada)
  - Verificacao da coluna `Ativa` (todas ATIVA)
  - Padronizacao UPPERCASE em todos os campos textuais
  - Criacao da coluna `empresa` no formato `PRIMEIRO_NOME (ICAO)`, ex: `GOL (GLO)`
  - Remocao da empresa com ICAO `***` (invalido)
  - Padronizacao snake_case nos nomes de coluna

### Padroes aplicados em todas as tabelas
- Nomes de coluna em **snake_case** (funcao `to_snake_case` definida na celula 51)
- Campos textuais em **UPPERCASE** com `trim`
- Tipos de dados adequados (int, double, timestamp, date)
- Persistencia no schema `mvp_engenharia_dados.silver` com `mode("overwrite")` e `overwriteSchema`
